# NB_04 · Développement des modèles

### Prédiction des annulations de réservations hôtelières

## Objectif du notebook

Ce notebook est consacré à l'entraînement des modèles de classification supervisée, à partir des jeux de données préparés dans le Notebook 03 (Prétraitement).

L'objectif est d'entraîner plusieurs algorithmes dans des conditions comparables, sur la version originale (556 variables) et sur la version réduite par PCA (9 composantes), afin de permettre une comparaison rigoureuse de leurs performances dans le Notebook 05 (Évaluation).

Les analyses portent notamment sur :

- le choix des algorithmes et de leurs hyperparamètres ;
- l'entraînement de chaque modèle sur les deux versions de données ;
- le chronométrage du temps d'entraînement ;
- la sauvegarde des modèles et d'un manifeste récapitulatif.


# 1. Chargement des données prétraitées

Les matrices sparse produites par le Notebook 03 sont chargées, ainsi que la cible d'entraînement.

In [1]:
from pathlib import Path
import time
import re
import joblib
import pandas as pd
from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_train_original = sparse.load_npz(PROCESSED_DIR / "X_train.npz")
X_train_pca = sparse.load_npz(PROCESSED_DIR / "X_train_pca.npz")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["is_canceled"]

print("Train original :", X_train_original.shape)
print("Train PCA      :", X_train_pca.shape)
print("Cible          :", y_train.shape)


Train original : (69782, 556)
Train PCA      : (69782, 9)
Cible          : (69782,)


### Interprétation

Les deux versions du jeu d'entraînement sont chargées avec succès : 69 782 réservations, réparties sur 556 variables pour la version originale et 9 composantes pour la version PCA. Ces dimensions correspondent exactement à celles produites en sortie du Notebook 03.

### Décision

Les deux matrices seront utilisées en parallèle pour entraîner chaque algorithme, afin de permettre une comparaison directe de leurs performances selon la version de données utilisée.

# 2. Définition des modèles

Quatre algorithmes de classification sont sélectionnés, chacun décliné en deux versions (Original et PCA), soit huit modèles au total. Les hyperparamètres principaux sont maintenus identiques entre les deux versions, afin que la comparaison porte uniquement sur l'effet de la PCA et non sur des différences de configuration.

In [2]:
model_specs = [
    ("Logistic Regression", "Original", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, solver="liblinear"
    )),
    ("Decision Tree - Entropy", "Original", DecisionTreeClassifier(
        criterion="entropy", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Decision Tree - Gini", "Original", DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Random Forest", "Original", RandomForestClassifier(
        n_estimators=20, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced_subsample", min_samples_leaf=3,
        max_depth=18, max_features="sqrt"
    )),
    ("Logistic Regression - PCA", "PCA", LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, solver="liblinear"
    )),
    ("Decision Tree - Entropy - PCA", "PCA", DecisionTreeClassifier(
        criterion="entropy", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Decision Tree - Gini - PCA", "PCA", DecisionTreeClassifier(
        criterion="gini", random_state=RANDOM_STATE, max_depth=3,
        min_samples_leaf=5, class_weight="balanced"
    )),
    ("Random Forest - PCA", "PCA", RandomForestClassifier(
        n_estimators=20, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced_subsample", min_samples_leaf=3,
        max_depth=18, max_features="sqrt"
    ))
]

print("Nombre de modèles à entraîner :", len(model_specs))


Nombre de modèles à entraîner : 8


### Interprétation

Quatre familles d'algorithmes sont retenues, pour un total de 8 configurations : une Logistic Regression (modèle linéaire de référence), deux Decision Tree utilisant des critères de division différents (Entropy et Gini), et un Random Forest (méthode d'ensemble). Les arbres sont volontairement limités à une profondeur de 3 niveaux pour rester interprétables, et le paramètre `class_weight="balanced"` (ou `"balanced_subsample"` pour la forêt) compense le déséquilibre des classes observé dans l'EDA.

### Décision

Ce choix d'algorithmes couvre un éventail représentatif des approches de classification (linéaire, arbre simple, ensemble d'arbres), avec des hyperparamètres raisonnables et comparables entre versions Original et PCA. Ce choix méthodologique — comparer les deux versions de données à hyperparamètres égaux — permettra d'isoler l'effet réel de la réduction de dimension dans le Notebook 05.

# 3. Entraînement, chronométrage et sauvegarde

Chaque modèle est entraîné sur la version de données qui lui correspond (Original ou PCA), avec mesure du temps d'entraînement et sauvegarde du modèle entraîné.

In [3]:
training_rows = []

for name, dataset_version, model in model_specs:
    X_fit = X_train_pca if dataset_version == "PCA" else X_train_original
    print(f"Entraînement : {name} ({dataset_version})")

    start = time.perf_counter()
    model.fit(X_fit, y_train)
    elapsed = time.perf_counter() - start

    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    path = MODEL_DIR / f"{slug}.joblib"
    joblib.dump(model, path)

    training_rows.append({
        "Modèle": name,
        "Famille": name.replace(" - PCA", ""),
        "Version_données": dataset_version,
        "Fichier": path.name,
        "Temps_entraînement_secondes": elapsed,
        "Nombre_variables": X_fit.shape[1]
    })

training_manifest = pd.DataFrame(training_rows)
assert training_manifest["Temps_entraînement_secondes"].notna().all()
training_manifest.to_csv(MODEL_DIR / "model_manifest.csv", index=False)
training_manifest.round(4)


Entraînement : Logistic Regression (Original)


Entraînement : Decision Tree - Entropy (Original)
Entraînement : Decision Tree - Gini (Original)


Entraînement : Random Forest (Original)


Entraînement : Logistic Regression - PCA (PCA)
Entraînement : Decision Tree - Entropy - PCA (PCA)


Entraînement : Decision Tree - Gini - PCA (PCA)


Entraînement : Random Forest - PCA (PCA)


,Modèle,Famille,Version_données,Fichier,Temps_entraînement_secondes,Nombre_variables
0,Logistic Regression,Logistic Regression,Original,logistic_regression.joblib,1.6159,556
1,Decision Tree - Entropy,Decision Tree - Entropy,Original,decision_tree_entropy.joblib,0.1355,556
2,Decision Tree - Gini,Decision Tree - Gini,Original,decision_tree_gini.joblib,0.1191,556
3,Random Forest,Random Forest,Original,random_forest.joblib,2.6526,556
4,Logistic Regression - PCA,Logistic Regression,PCA,logistic_regression_pca.joblib,0.0980,9
5,Decision Tree - Entropy - PCA,Decision Tree - Entropy,PCA,decision_tree_entropy_pca.joblib,0.4080,9
6,Decision Tree - Gini - PCA,Decision Tree - Gini,PCA,decision_tree_gini_pca.joblib,0.3246,9
7,Random Forest - PCA,Random Forest,PCA,random_forest_pca.joblib,16.0309,9


### Interprétation

Les 8 modèles s'entraînent sans erreur. Le temps d'entraînement varie fortement selon l'algorithme et la version de données : les arbres de décision s'entraînent en une fraction de seconde (0,11 à 0,36 s), tandis que le Random Forest est nettement plus long, en particulier sur la version PCA (17,2 s contre 2,8 s pour la version originale). Ce résultat, à première vue contre-intuitif, s'explique par le fait que la PCA transforme les données en une matrice **dense** (les zéros du One-Hot Encoding disparaissent) : le Random Forest doit alors évaluer beaucoup plus de seuils de coupure continus sur seulement 9 variables denses, contre des divisions binaires triviales sur 556 colonnes creuses.

### Décision

Le temps d'entraînement à lui seul ne permet donc pas de conclure à l'intérêt de la PCA : la version PCA n'est pas systématiquement plus rapide, et la variable de comparaison réellement déterminante reste la performance prédictive, évaluée dans le Notebook 05. Le manifeste (`models/model_manifest.csv`) conserve l'ensemble de ces informations pour appuyer cette comparaison.

# 4. Conclusion

Les 8 modèles (4 algorithmes × 2 versions de données) ont été entraînés avec le même découpage entraînement/test et des hyperparamètres comparables entre versions. Le manifeste `models/model_manifest.csv` conserve, pour chaque modèle, sa famille d'algorithme, la version de données utilisée, le nombre de variables et le temps d'entraînement.

Le Notebook 05 (Évaluation) déterminera, à partir de métriques de performance calculées sur le jeu de test, si la forte réduction du nombre de prédicteurs obtenue par la PCA (556 → 9 variables) compense la perte potentielle de performance et d'interprétabilité qu'elle implique.
